# Data preparation

We start by importing the already "enriched" dataset to clean it. By enriched we mean that some data has already been re-integrated, as we explain in detail in the project's documentation.

Here we study the datasets composition.

In [ ]:
import pandas as pd

tracks = pd.read_csv("../enriched_datasets/tracks_enriched.csv")
artists = pd.read_csv("../enriched_datasets/artists.csv")

tracks = tracks.copy()
artists = artists.copy()

print(f"Initial dataset info:\n")
tracks.info()
artists.info()

# Print original number of tracks and artists
print(f"Tracks shape: {tracks.shape[0]} rows x {tracks.shape[1]} columns")
print(f"Artists shape: {artists.shape[0]} rows x {artists.shape[1]} columns")

## Casting tracks into correct types

In [ ]:
# Objects to strings
columns_to_string_tracks = ["id", "id_artist", "name_artist", "full_title", "title", "featured_artists", "primary_artist", "language", "album", "album_name", "album_type", "lyrics", "album_image", "id_album"]
for column in columns_to_string_tracks:
    tracks[column] = tracks[column].astype("string")

# Album release date: object -> datetime
tracks["album_release_date"] = pd.to_datetime(tracks["album_release_date"], errors="coerce")

# Floats/Objects to Ints
columns_to_ints_tracks = ["year", "month", "day", "popularity"]
for column in columns_to_ints_tracks:
    tracks[column] = pd.to_numeric(tracks[column], errors="coerce")
    tracks[column] = tracks[column].astype("Int64")

# Explicit: object -> boolean
print(tracks["explicit"].head())
tracks["explicit"] = tracks["explicit"].astype("bool")

print(f"Tracks after correct casting:\n")
tracks.info()

## Casting artists into correct types

In [ ]:
columns_to_string_artists = ["id_author", "name", "gender", "birth_place", "nationality", "description", "province", "region", "country", "source"]
for column in columns_to_string_artists:
    artists[column] = artists[column].astype("string")
    
columns_to_datetime_artists = ["birth_date", "active_start", "active_end"]
for column in columns_to_datetime_artists:
    artists[column] = pd.to_datetime(artists[column], errors='coerce')

print(f"Artists after correct casting:\n")
artists.info()

## Removing duplicates:
As the _id_ of the track will not be useful later in the analysis, we check if the dataset has duplicates by doing:
- a check on whether or not the _id\_author_ (in the artists dataset) is duplicated;
- a check on whether, inside the tracks dataset, there are tracks with both the same _id\_artist_ and _full\_title_.

We then remove every duplicate and keep only the first occurrence.

In [ ]:
# IDs appearing more than once
duplicated_ids = artists["id_author"].value_counts()[artists["id_author"].value_counts() > 1]
print(f"Dupicated ids:\n", duplicated_ids)

duplicate_pairs = tracks[tracks.duplicated(subset=["id_artist", "full_title"], keep=False)]
print(f"Duplicated pairs:\n", duplicate_pairs)

# Keeping only first ID and full_title of the duplicates
tracks = tracks.drop_duplicates(subset=["id_artist", "full_title"], keep="first")

print(f"Tracks shape after removing duplicates: {tracks.shape[0]} rows x {tracks.shape[1]} columns")
print(f"Artists shape after removing duplicates: {artists.shape[0]} rows x {artists.shape[1]} columns")

## Removing non-numeric features
With the clustering goal in mind, we decided to remove non-numeric columns with the following exceptions:
- keeping _id\_artist_ and _id\_author_ to later merge the two datasets;
- keeping _datetime_ columns to later extract only the day, month or year;
- keeping _booleans_ to later map them into numbers;
- keeping _language_ as it could be interesting for any subsequent analysis.

This step should be performed **before nans removal**, to avoid removing rows for a non significant cause.

In [ ]:
# TRACKS
numeric_cols_t = tracks.select_dtypes(include=["number"]).columns
for col in numeric_cols_t:
    tracks[col] = pd.to_numeric(tracks[col], errors='coerce')

reduced_tracks = tracks[numeric_cols_t]

# Inserting artist id (string)
if "id_artist" in tracks.columns:
    reduced_tracks.insert(0, "id_artist", tracks["id_artist"])

# Inserting album_release (datetime)
if "album_release_date" in tracks.columns:
    reduced_tracks.insert(1, "album_release_date", tracks["album_release_date"])

# Inserting explicit, modified_popularity (bool)
if "explicit" in tracks.columns:
    reduced_tracks.insert(2, "explicit", tracks["explicit"])

if "modified_popularity" in tracks.columns:
    reduced_tracks.insert(3, "modified_popularity", tracks["modified_popularity"])

# Inserting language (string)
if "language" in tracks.columns:
    reduced_tracks.insert(4, "language", tracks["language"])

print(f"Tracks shape after removing non-numeric features: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

In [ ]:
# ARTISTS
numeric_cols_a = artists.select_dtypes(include=["number"]).columns
for col in numeric_cols_a:
    artists[col] = pd.to_numeric(artists[col], errors='coerce')

reduced_artists = artists[numeric_cols_a]

# Inserting author id
if "id_author" in artists.columns:
    reduced_artists.insert(0, "id_author", artists["id_author"])

# Inserting datetimes
if "birth_date" in artists.columns:
    reduced_artists.insert(1, "birth_date", artists["birth_date"])

if "active_start" in artists.columns:
    reduced_artists.insert(2, "active_start", artists["active_start"])

if "active_end" in artists.columns:
    reduced_artists.insert(3, "active_end", artists["active_end"])

print(f"Artists shape after removing non-numeric features: {reduced_artists.shape[0]} rows x {reduced_artists.shape[1]} columns")

## Nan values management

From now on we work with already reduced datasets. For both tracks and artists datasets, we use some strategies to avoid the elimination of entire columns or too many entries.

Let's look at them separately.

### Tracks nan values
To avoid either removing the _album\_release\_date_ column or losing more than 300 entries, we extracted the year from the entire date, and replaced the nan values with the _year_ value of the same track. This behaviour is justified if we think that some songs could have been released as singles.

We then drop the _album\_release\_date_ feature, keeping only the _album\_release\_year_.

We arbitrarely decided to keep only the columns with less than 300 nans, and in the end we removed all the remaining rows containing at least one nan.

In [ ]:
reduced_tracks["album_release_year"] = (
    reduced_tracks["album_release_date"].dt.year
    .fillna(reduced_tracks["year"])
)

reduced_tracks = reduced_tracks.drop(columns=["album_release_date"])

tracks_nan_per_feature = reduced_tracks.isna().sum()
print(f"Nan per feature (tracks):\n", tracks_nan_per_feature)

tracks_nan_per_row = reduced_tracks.isna().sum(axis=1)
print(f"Nan per row:\n", tracks_nan_per_row)

# Removing columns with more than 300 nans
cols_to_drop_tracks = tracks_nan_per_feature[tracks_nan_per_feature > 300].index
reduced_tracks = reduced_tracks.drop(columns=cols_to_drop_tracks)

print("Dropped columns for tracks:", list(cols_to_drop_tracks))

# Dropping rows with at least one nan
reduced_tracks = reduced_tracks.dropna()

# Checking remaining tracks
print(f"Tracks shape after removing nans: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

### Artists nan values
Similarly to the case above, as we have many missing values in the _active\_start_ column, we decided to replace those values with the year of pubblication of the oldest song from the same artist. As before, we only kept the year as an _active\_start\_year_ feature.

Here we removed only the columns with more than 20 nans; in the end we removed all rows with at least one nan.

In [ ]:
# Active start year
first_year_by_artist = reduced_tracks.groupby("id_artist")["year"].min()
first_year_by_artist.name = "first_track_year"

print(first_year_by_artist.head())

reduced_artists = reduced_artists.merge(
    first_year_by_artist,
    left_on="id_author",
    right_on="id_artist",
    how="left"
)

print(reduced_artists.columns)

reduced_artists["active_start_year"] = (
    reduced_artists["active_start"].dt.year
    .fillna(reduced_artists["first_track_year"])
)

reduced_artists = reduced_artists.drop(columns=["active_start"])


artists_nan_per_feature = reduced_artists.isna().sum()
print(f"Nan per feature (artists):\n", artists_nan_per_feature)

artists_nan_per_row = reduced_artists.isna().sum(axis=1)
print(f"Nan per row (artists):\n", artists_nan_per_row)

# Removing columns with more than 20 nans
cols_to_drop_artists = artists_nan_per_feature[artists_nan_per_feature > 20].index
reduced_artists = reduced_artists.drop(columns=cols_to_drop_artists)

print("Dropped columns for artists:", list(cols_to_drop_artists))

# Dropping rows with at least one nan
reduced_artists = reduced_artists.dropna()

#Checking remaining artists
print(f"Artists shape after removing nans: {reduced_artists.shape[0]} rows x {reduced_artists.shape[1]} columns")

In [ ]:
# Final check on all nans (should both be zero)
print(f"Making sure we dropped nans:\n")

tracks_nan_per_feature = reduced_tracks.isna().sum()
print(f"Nan per feature after nan removal(tracks):\n", tracks_nan_per_feature)

artists_nan_per_feature = reduced_artists.isna().sum()
print(f"Nan per feature after nan removal(artists):\n", artists_nan_per_feature)


## Invalid values removal
For all the features with known domain, we removed the invalid values.

We performed this operation on both tracks and artists features.

All domain choises will be explained in detail in the project's document:
- **popularity** → [0, 100];
- **explicit** → we correct the explicit labeling by setting it _True_ if the swear words count (both english and italian) is non-zero;
- **year**, **album_release_year** → <= 2025 (should not be in the future!);
- **zcr**, **flatness** → [0, 1]
- **active_start_year** > **birth_date_year**
- **latitude** → [35, 47]
- **longitude** → [4, 19]

As you can see, we don't act upon every incorrect value: if artist have invalid (_latitude_, _longitude_), they could be born outside of Italy and still have Italian citizenship.

Some checks are only made to raise attention on unexpected values.

Indeed, this is also what we did with the last prints of the following cell: we look at min and max values of these feature, just to see if they are reasonable or interesting in any way.

As we can see, the _modified\_popularity_ feature seems to be uninteresting, so we drop its column.

In [ ]:
# popularity
reduced_tracks = reduced_tracks[
    (reduced_tracks["popularity"] >= 0) &
    (reduced_tracks["popularity"] <= 100)
]

print(f"Checking min popularity:", reduced_tracks["popularity"].min())
print(f"Checking max popularity:", reduced_tracks["popularity"].max())

print(f"Tracks shape after removing invalid popularity: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

In [ ]:
# swear_IT
count_invalid_swear_IT = (reduced_tracks["swear_IT"] < 0).sum()
print(f"Invalid swear_IT counter:", count_invalid_swear_IT)

print("\n")

# swear_EN
count_invalid_swear_EN = (reduced_tracks["swear_EN"] < 0).sum()
print(f"Invalid swear_EN counter:", count_invalid_swear_EN)

# explicit
count_swear = ((reduced_tracks["swear_IT"] > 0) | (reduced_tracks["swear_EN"] > 0)).sum()
print(f"Number of rows with swear words:", count_swear)

count_explicit = reduced_tracks["explicit"].sum()
print(f"Counter of explicit tracks:", count_explicit)

reduced_tracks["explicit"] = (
    (reduced_tracks["swear_IT"] > 0) |
    (reduced_tracks["swear_EN"] > 0)
).astype(int)

print(f"Counter of explicit tracks after correction:", reduced_tracks["explicit"].sum())

In [ ]:
# year
reduced_tracks = reduced_tracks[reduced_tracks["year"] <= 2025]

print(f"Tracks shape after removing invalid year: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

print("\n")

# album_release_year
reduced_tracks = reduced_tracks[reduced_tracks["album_release_year"] <= 2025]

print(f"Tracks shape after removing invalid album release year: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

In [ ]:
# zcr
count_invalid_zcr = ((reduced_tracks["zcr"] < 0) | (reduced_tracks["zcr"] > 1)).sum()
print(f"Invalid number of zcr:", count_invalid_zcr)

print("\n")

# flatness
count_invalid_flatness = ((reduced_tracks["flatness"] < 0) | (reduced_tracks["flatness"] > 1)).sum()
print(f"Invalid number of flatness:", count_invalid_flatness)

In [ ]:
# active_start_year
reduced_artists["birth_date_year"] = (
    reduced_artists["birth_date"].dt.year
)

reduced_artists = reduced_artists.drop(columns=["birth_date"])

count_invalid_active_start = (reduced_artists["active_start_year"] <  reduced_artists["birth_date_year"]).sum()
print(f"Invalid number of active start year:", count_invalid_active_start)

mask_invalid_active_start = reduced_artists["active_start_year"] < reduced_artists["birth_date_year"]

reduced_artists.loc[mask_invalid_active_start, "active_start_year"] = (
    reduced_artists["birth_date_year"] + 10
)

print("\n")

# latitude
count_invalid_latitude = ((reduced_artists["latitude"] < 35) | (reduced_artists["latitude"] > 47)).sum()
print(f"Invalid number of latitude:", count_invalid_latitude)

invalid_latitude = reduced_artists[
    (reduced_artists["latitude"] <= 35) |
    (reduced_artists["latitude"] >= 47)
]

print(f"Artists with invalid latitude:", invalid_latitude[["id_author", "latitude", "longitude"]])
print("Checked in dataset: born in Santo Domingo, italian.")

print("\n")

# longitude
count_invalid_longitude = ((reduced_artists["longitude"] < 4) | (reduced_artists["longitude"] > 19)).sum()
print(f"Invalid number of longitude:", count_invalid_longitude)

invalid_longitude = reduced_artists[
    (reduced_artists["longitude"] <= 4) |
    (reduced_artists["longitude"] >= 19)
]

print(f"Artists with invalid longitude:", invalid_longitude[["id_author", "latitude", "longitude"]])
print("Checked in dataset: born in Santo Domingo, italian.")

In [ ]:
# disc_number
print(f"Disc number min value:", reduced_tracks["disc_number"].min())
print(f"Disc number max value:", reduced_tracks["disc_number"].max())

print("\n")

#track_number
print(f"Track number min value:", reduced_tracks["track_number"].min())
print(f"Track number max value:", reduced_tracks["track_number"].max())

print("\n")

#duration_ms
print(f"Duration min value:", reduced_tracks["duration_ms"].min())
print(f"Duration max value:", reduced_tracks["duration_ms"].max())

print("\n")

# modified_popularity
print(f"Modified popularity min value:", reduced_tracks["modified_popularity"].min())
print(f"Modified popularity max value:", reduced_tracks["modified_popularity"].max())
print("Not useful. Dropping column.")

reduced_tracks = reduced_tracks.drop(columns=["modified_popularity"])

## Outliers removal:
We divided outliers removal into four main categories. In each we made different decisions.

### Sound features
As we know that all the following **sound features** have a domain $[0, \inf)$, we first made sure the lower bound wasn't negative.

We then approached the outlier removal using the sigmas technique, more malleable than the IQR technique for this specific case.

### Year features for tracks
As for the _year_ and _album\_release\_year_, having already defined the upper bound, we applied the outliers removal to the lower bound only.

### Lexical features
We applied the same methodology, first making sure that all the values are positive, then exploring the acceptable domain of each feature.

Taking into account the possibility of having longer tracks (such as rap-ballads), or tracks in which a single character could be considered as a word, we decided not to cut some of the outliers as they may be interesting for our clustering.

We looked at the min and max values of the feature to determine which ones could be affected by stronger outliers, and then chose to cut them accordingly.

### Year features for artists
We evaluated the outliers for this last column and _active\_start\_year. As all the dates seamed reasonable, we decided to not perform any cut.

In [ ]:
# === 1. ===
sound_outlier_mask_total = pd.Series(False, index=reduced_tracks.index)

sound_cols = ["bpm", "centroid", "rolloff", "flux", "rms", "spectral_complexity", "pitch", "loudness"]
for feature in sound_cols:
    col = reduced_tracks[feature]

    min = col.min()
    max = col.max()
    median = col.median()
    mu = col.mean()
    sigma = col.std()

    lower = mu - 4 * sigma
    upper = mu + 4 * sigma

    outlier_mask = (col < lower) | (col > upper)
    n_outliers = outlier_mask.sum()

    print("="*40)
    print(f"Feature: {feature}")
    print(f"Min value: {min}")
    print(f"Max value: {max}")
    print(f"Median value:{median}")
    print(f"Acceptable domain (4*sigma): [{lower:.3f}, {upper:.3f}]")
    print(f"Outliers found: {n_outliers}")

    sound_outlier_mask_total |= outlier_mask

# Counts total of outliers
print("Sound outliers rows:", sound_outlier_mask_total.sum())

# Removing outliers rows
reduced_tracks = reduced_tracks[~sound_outlier_mask_total]

print("\n")
print(f"Tracks shape after sound outliers removal: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")
print("\n")

In [ ]:
# === 2. ===
tracks_year_outlier_mask_total = pd.Series(False, index=reduced_tracks.index)

tracks_datetime_cols = ["year", "album_release_year"]
for feature in tracks_datetime_cols:
    col = reduced_tracks[feature]

    min = col.min()
    max = col.max()
    median = col.median()
    mu = col.mean()
    sigma = col.std()

    lower = mu - 5 * sigma
    upper = 2025

    outlier_mask = (col < lower) | (col > upper)
    n_outliers = outlier_mask.sum()

    print("="*40)
    print(f"Feature: {feature}")
    print(f"Min value: {min}")
    print(f"Max value: {max}")
    print(f"Median value:{median}")
    print(f"Acceptable domain (5*sigma): [{lower:.3f}, {upper:.3f}]")
    print(f"Outliers found: {n_outliers}")

    tracks_year_outlier_mask_total |= outlier_mask

# Counts total of outliers
print("Tracks year outliers rows:", tracks_year_outlier_mask_total.sum())

# Removing outliers rows
reduced_tracks = reduced_tracks[~tracks_year_outlier_mask_total]

print("\n")
print(f"Tracks shape after year outliers removal: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")
print("\n")

In [ ]:
# === 3. ===
lexical_cols = ["n_sentences", "n_tokens", "tokens_per_sent", "char_per_tok", "lexical_density", "avg_token_per_clause"]

# Choosing only some columns to clean
cols_to_clean = ["tokens_per_sent", "avg_token_per_clause"]

for feature in lexical_cols:
    col = reduced_tracks[feature]

    mu = col.mean()
    sigma = col.std()
    lower = mu - 5 * sigma
    upper = mu + 5 * sigma

    outlier_mask = (col < lower) | (col > upper)
    n_outliers = outlier_mask.sum()

    print("="*40)
    print(f"Feature: {feature}")
    print(f"Min value: {col.min()}")
    print(f"Max value: {col.max()}")
    print(f"Median value: {col.median()}")
    print(f"Acceptable domain (5*sigma): [{lower:.3f}, {upper:.3f}]")
    print(f"Outliers found: {n_outliers}")

    # Outlier removal only on chosen columns
    if feature in cols_to_clean:
        reduced_tracks = reduced_tracks[~outlier_mask]
        print(f"Removed {n_outliers} outliers from {feature}")
    else:
        print("Skipped outlier removal for this feature.")

print("\n")
print(f"Tracks shape after lexical outliers removal: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")
print("\n")

In [ ]:
# === 4. ===
artists_datetime_cols = ["birth_date_year", "active_start_year"]
for feature in artists_datetime_cols:
    col = reduced_artists[feature]

    min = col.min()
    max = col.max()
    median = col.median()
    mu = col.mean()
    sigma = col.std()

    lower = mu - 5 * sigma
    upper = 2025

    outlier_mask = (col < lower) | (col > upper)
    n_outliers = outlier_mask.sum()

    print("="*40)
    print(f"Feature: {feature}")
    print(f"Min value: {min}")
    print(f"Max value: {max}")
    print(f"Median value:{median}")
    print(f"Acceptable domain (5*sigma): [{lower:.3f}, {upper:.3f}]")
    print(f"Outliers found: {n_outliers}")

print("\n")
print("No need to remove anything.")

## Assessment of the final datasets
We print each dataset info before saving them.

In [ ]:
print(f"Tracks shape after data preparation: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")
print(f"Artists shape after after data preparation: {reduced_artists.shape[0]} rows x {reduced_artists.shape[1]} columns")

print(f"Tracks sample:\n", reduced_tracks.head())
print(f"Artists sample:\n", reduced_artists.head())

print(f"Tracks types:", reduced_tracks.info())
print(f"Artists types:", reduced_artists.info())